# CodeBlooded v1: train the model on Kaggle (owner: Vaishnavi)

**Settings (right panel) before you run anything**
1. Accelerator: **GPU T4 x2** (not P100: recent PyTorch builds may not support it).
2. Internet: **On** (only needed here to clone the repo and install packages; the judged server never uses internet).
3. Add Data: the Kaggle Dataset with `india_climate_train.nc` and `stats.json` from the kit's v1.0.0 release.

**What you send back to Bhoomi:** `/kaggle/working/codeblooded_v1_bundle.zip` (bundle + results + training log). Nothing else.

In [ ]:
# 1. Configuration: change only these lines
REPO = "https://github.com/Bhoomified/CodeBlooded.git"
BRANCH = "main"                    # train from main after PR #2 (backend) is merged
WIDTH, BLOCKS = 96, 4              # model size (width 96 = about 0.7M parameters)
EPOCHS = 30                        # single-step epochs
ROLLOUT = "4:8:1e-4,8:4:3e-5"      # steps:epochs:learning-rate for the rollout phases

In [ ]:
# 2. Get the code at a known commit
!rm -rf CodeBlooded && git clone --depth 1 -b $BRANCH $REPO
%cd CodeBlooded
!git log --oneline -1 | tee /kaggle/working/commit.txt

In [ ]:
# 3. Install what Kaggle lacks, check the GPU
!pip install -q netCDF4 "monsoon-r1-kit @ git+https://github.com/OfficialAbhinavSingh/yug-monsoon-kit@v1.0.0"
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
# 4. Find the data and run the data checks (handbook tests A1-A2: channel order, latitude direction, no NaN)
import glob, xarray as xr, numpy as np
DATA = glob.glob("/kaggle/input/**/india_climate_train.nc", recursive=True)[0]
STATS = glob.glob("/kaggle/input/**/stats.json", recursive=True)[0]
ds = xr.open_dataset(DATA)
print(DATA, ds["data"].shape, str(ds.time.values[0])[:10], "->", str(ds.time.values[-1])[:10])
assert list(ds["channel"].values) == ["t2m", "tp", "u10", "v10", "msl", "tcwv"]
assert ds["lat"].values[0] < ds["lat"].values[-1], "latitude must run south to north"
assert np.isfinite(ds["data"].values).all()
print("data checks passed")

In [ ]:
# 5. Smoke test (2-3 minutes): proves the whole pipeline runs before the long job
!python train/pipeline.py --data "$DATA" --out /kaggle/working/bundle_smoke --quick

In [ ]:
# 6. Full training (the long step). The log is saved; a checkpoint is written after every phase (cnn_partial.pt)
!python train/pipeline.py --data "$DATA" --out /kaggle/working/bundle --device cuda \
    --width $WIDTH --blocks $BLOCKS --epochs $EPOCHS --rollout "$ROLLOUT" 2>&1 | tee /kaggle/working/train.log

In [ ]:
# 7. Score exactly like the grader on held-out 2020-21 and print the ship gates
!python eval/local_score.py --data "$DATA" --stats "$STATS" --bundle /kaggle/working/bundle \
    --out /kaggle/working/results.json 2>&1 | tee /kaggle/working/score.log

In [ ]:
# 8. Short summary to paste into the team chat
import json
r = json.load(open("/kaggle/working/results.json"))
for x in r:
    print(f"{x['model']:42s} total {x['total_auto_70']:6.2f}/70  skill {x['mean_skill']:+.3f}  csi {x['mean_csi_skill']:+.3f}  day4>persistence {x['beats_persistence_day4']}")

In [ ]:
# 9. Package for Bhoomi: bundle (without the partial checkpoint) + results + logs + commit id
!cd /kaggle/working && rm -f bundle/cnn_partial.pt codeblooded_v1_bundle.zip && \
  zip -r codeblooded_v1_bundle.zip bundle results.json train.log score.log commit.txt && ls -la bundle codeblooded_v1_bundle.zip